## Compulsory submission checklist (hackathon / rubric)

Use this section so nothing required is missed. **Fill in your public URLs** after you push to GitHub and (if needed) upload a copy to Colab.

| # | Requirement | Status | Your link / note |
|---|-------------|--------|------------------|
| 1 | **Public GitHub repo** with this notebook at `notebooks/ReleaseOps_final_walkthrough.ipynb` | ☐ | `https://github.com/eshwanthkartitr/RL` |
| 2 | **Google Colab** — open the same notebook (must exist on `main`). | ☐ | `https://colab.research.google.com/github/eshwanthkartitr/RL/blob/main/notebooks/ReleaseOps_final_walkthrough.ipynb` |
| 3 | **Hugging Face Space** (live environment / API) | ☐ | `https://huggingface.co/spaces/hiitsesh/New_gpu_space` |
| 4 | **Demo or pitch video** (if required) | ☐ | YouTube or other public URL |
| 5 | **README** lists Space + Colab + video | ☐ | Edit [README.md](../README.md) |

**How to get a Colab link (one-time):**
1. Push this repository to **public** GitHub (notebook path must match the URL above).
2. If you forked/renamed the repo, update the `eshwanthkartitr/RL` segment in the Colab URL to match your `owner/repo` on GitHub.
3. Paste that full Colab URL into the submission form and in **README** → *Training / walkthrough* bullet.
4. *Alternative:* In Colab: **File → Upload notebook** → upload this `.ipynb` → **File → Save a copy in Google Drive** → **Share** → use that Drive link in the form (and README). Prefer the `colab.research.google.com/github/...` form when the repo is public so it stays in sync with Git.

**Next cell** (`REPO` setup) also prints a Colab link if you set `GITHUB_NOTEBOOK_USER` and `GITHUB_NOTEBOOK_REPO` in the environment.

# ReleaseOps RL — end-to-end walkthrough

**Purpose:** a single place for your demo content: (1) **dataset** generation, (2) **GRPO training**, (3) **load the finetuned checkpoint** and **measure** behavior vs baseline.

**Prerequisites:** this repo, Python with `transformers`, `trl`, `torch`, `requests`, and the rest of `requirements.txt`. Run Jupyter **after** `cd` to the project root, or set `REPO` in the first code cell to your clone path.

**Hackathon / Colab: what to show**  
* **Sufficient for many judges:** the **live Hugging Face Space** (API + training endpoints) *plus* this repo link — you do *not* have to re-run a full 100-step training live if you already have plots/metrics on disk or in the README.  
* **Stronger story:** Space for “there is a real environment” + **one** of: (a) a training curve / `metrics` JSON, (b) a short `evaluate_llm_baseline` run from this notebook, (c) `compare_qwen17_eval.sh` outputs.  
* **Note:** the Space **deployment** is built from the `New_gpu_space` Hub project; *this* notebook uses the **`Boiler_plate` / clone** paths (`training/…`). Same logic, different folder layout on Hub.

**Open in Colab:** open this file from your GitHub clone (File → upload, or `https://github.com/<you>/<repo>/blob/main/notebooks/ReleaseOps_final_walkthrough.ipynb`) and set the first code cell’s working directory to the clone root.


In [ ]:
# Resolve repo root (works if cwd is repo root, `notebooks/`, or a parent of `training/`)
import os
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
for _ in range(4):
    if (REPO / "training" / "make_dataset.py").exists():
        break
    REPO = REPO.parent
else:
    raise SystemExit("Could not find training/make_dataset.py — chdir to Boiler_plate or set REPO manually.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if str(REPO / "training") not in sys.path:
    sys.path.insert(0, str(REPO / "training"))

# Colab (and some containers): HOME may be / or missing, so HuggingFace/datasets can try to use /.cache (PermissionError). Point caches under the repo or /content.
_in_colab = (Path("/content").exists() and str(REPO).startswith("/content")) or bool(os.environ.get("COLAB_RELEASEOPS", ""))
if _in_colab or not (os.environ.get("HF_HOME") or "").strip():
    hf = REPO / ".cache" / "huggingface"
    hf.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("HF_HOME", str(hf))
    h = os.environ["HF_HOME"]
    os.environ.setdefault("HF_DATASETS_CACHE", str(Path(h) / "datasets"))
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(Path(h) / "hub"))
    os.environ.setdefault("TRANSFORMERS_CACHE", str(Path(h) / "transformers"))
if (os.environ.get("HOME") or "").strip() in ("", "/"):
    os.environ["HOME"] = str(REPO / ".cache" / "user_home")
    Path(os.environ["HOME"]).mkdir(parents=True, exist_ok=True)

print("REPO =", REPO)
if _in_colab or os.environ.get("HF_HOME"):
    print("HF_HOME =", os.environ.get("HF_HOME"), "| HOME =", os.environ.get("HOME"))

_u = (os.environ.get("GITHUB_NOTEBOOK_USER") or "").strip()
_r = (os.environ.get("GITHUB_NOTEBOOK_REPO") or "").strip()
_b = (os.environ.get("GITHUB_NOTEBOOK_BRANCH") or "main").strip()
if _u and _r:
    print(
        "Open in Colab (for submission):",
        f"https://colab.research.google.com/github/{_u}/{_r}/blob/{_b}/notebooks/ReleaseOps_final_walkthrough.ipynb",
    )


## 1. Data generation

`training/make_dataset.py` writes JSONL **prompt + scenario** rows (`family`, `seed`, `difficulty`, `archetype_mix`) for GRPO. It creates:

| File | Role |
|------|------|
| `training/data/train.jsonl` | Training |
| `training/data/eval_seen.jsonl` / `eval_unseen.jsonl` | Held-out |
| `training/data/eval.jsonl` | Merged eval (for offline scripts) |

Regenerate with the next cell, or use the **existing** files if you do not need to resample.


In [ ]:
# Regenerate all splits (overwrites). Reduce counts only if you edit make_dataset.py __main__.
import subprocess, sys
r = subprocess.run(
    [sys.executable, str(REPO / "training" / "make_dataset.py")],
    cwd=REPO,
    capture_output=True,
    text=True,
)
print(r.stdout, r.stderr, sep="")
r.check_returncode()


In [ ]:
# Peek at one training row
import json
p = REPO / "training/data/train.jsonl"
line = p.read_text().splitlines()[0]
row = json.loads(line)
print(json.dumps(row, indent=2)[:2000], "…" if len(line) > 2000 else "")
print("families in train (first 20 lines):", {json.loads(x)["family"] for x in p.read_text().splitlines()[:20]})


## 2. Training (GRPO)

`training/train_grpo.py` runs **Group Relative Policy Optimization** with `ReleaseOpsGRPOEnv` (same tools and rewards as the hosted arena).

* **Full run** — set `--model-name` (e.g. `Qwen/Qwen3-1.7B`), `--output-dir`, `--max-steps`, etc. Training can take a long time and needs a suitable GPU for larger models.
* **Smoke** — `python training/train_grpo.py --smoke` uses tiny `max_steps` and a `*-smoke` output directory for a quick **pipeline** check.

The trainer can auto-run `make_dataset` if the train file is missing (see `ensure_dataset`).


In [ ]:
# Option A: quick smoke (uncomment to run; may need GPU + compatible TRL OpenEnv)
# import subprocess, sys
# subprocess.run(
#     [sys.executable, "training/train_grpo.py", "--smoke", "--model-name", "Qwen/Qwen2.5-0.5B-Instruct"],
#     cwd=REPO,
#     check=True,
# )

# Option B: print the full training command to copy to a machine with GPU
MODEL = "Qwen/Qwen3-1.7B"  # change to your base model
OUT = "outputs/releaseops-grpo-pilot"
cmd = f'''cd "{REPO}" && python training/train_grpo.py \
  --model-name {MODEL} \
  --output-dir {OUT} \
  --max-steps 50 \
  --train-file training/data/train.jsonl \
  --metrics-json outputs/grpo_env_metrics_pilot.json'''
print(cmd)


## 3. Load the finetuned model and check performance

**On disk (subfolder `best_by_loss`):** point HuggingFace at your repo root and use `from_pretrained(..., subfolder=)` **or** pass the subfolder’s path as the model id.

**From the Hub** (e.g. `hiitsesh/releaseops-grpo-1.7b-best`): same idea with `subfolder="best_by_loss"`.

**Metrics:** `training/evaluate_llm_baseline.py` runs the same tool-calling loop as the paper-style eval and writes JSON (`safe_ship`, `unsafe_ship`, `false_blocks`, `repaired_actions`, …). Use a **small** `--limit` for a quick check in the notebook; use `compare_qwen17_eval.sh` for a full **base vs finetuned** side-by-side on disk/Hub.


In [ ]:
# Load tokenizer + model (no generation — sanity check; uses ~4–7GB+ depending on model + dtype)
import os

# Set one of: local subfolder, or "org/repo" for Hub
CHECKPOINT = os.environ.get("RELEASEOPS_CKPT", "Qwen/Qwen3-1.7B")
SUBFOLDER = os.environ.get("RELEASEOPS_SUBFOLDER", "").strip() or None  # e.g. "best_by_loss" for Hub

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

load_kw = {"subfolder": SUBFOLDER} if SUBFOLDER else {}
tok = AutoTokenizer.from_pretrained(CHECKPOINT, **load_kw, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    torch_dtype=torch.float16,
    device_map="auto" if torch.cuda.is_available() else None,
    **load_kw,
    trust_remote_code=True,
)
if torch.backends.mps.is_available() and not torch.cuda.is_available():
    model = model.to("mps")
dev = next(model.parameters()).device
print("loaded:", CHECKPOINT, "| subfolder:", SUBFOLDER)
print("dtype:", next(model.parameters()).dtype, "| device:", dev)
if dev.type == "cpu":
    print("(CPU inference for 1.7B+ is slow; use GPU or skip this cell if you only need eval script.)")


In [ ]:
# Run a short eval (zero-shot or guided) and show metrics JSON
# Adjust --torch-model / --torch-subfolder to your finetuned Hub path or local best_by_loss directory.
import subprocess, json, os, sys, textwrap

# Defaults: your published repo + subfolder; override with env vars
TORCH = os.environ.get("RELEASEOPS_TORCH", "Qwen/Qwen3-1.7B")
SUB = (os.environ.get("RELEASEOPS_SUBFOLDER", "") or "").strip()
LIMIT = os.environ.get("RELEASEOPS_LIMIT", "3")
out_json = REPO / "outputs" / "nb_eval_result.json"
out_json.parent.mkdir(parents=True, exist_ok=True)

args = [
    str(REPO / "training/evaluate_llm_baseline.py"),
    "--backend", "torch",
    "--torch-model", TORCH,
    "--limit", str(LIMIT),
    "--output-json", str(out_json),
    "--eval-mode", "guided_zero_shot",
]
if SUB:
    args.extend(["--torch-subfolder", SUB])

cmd = " ".join(f'"{a}"' if " " in a else a for a in [sys.executable, *args])
print("Command:\n", textwrap.fill(cmd, 100))
# Uncomment the next 3 lines to actually run (can take a long time per step on CPU)
# r = subprocess.run([sys.executable, *args], cwd=REPO, capture_output=True, text=True)
# print(r.stdout); print(r.stderr, file=sys.stderr)
# r.check_returncode()

if out_json.exists():
    print(json.dumps(json.loads(out_json.read_text()), indent=2))
else:
    print("No", out_json, "yet — uncomment subprocess.run above after setting TORCH / SUBFOLDER / LIMIT.")


In [ ]:
# Optional: one-episode "answers only" (executed tool calls as JSONL)
import os, sys, subprocess, textwrap
TORCH = os.environ.get("RELEASEOPS_TORCH", "Qwen/Qwen3-1.7B")
SUB = (os.environ.get("RELEASEOPS_SUBFOLDER", "") or "").strip()
cmd = [
    sys.executable, str(REPO / "training/run_inference.py"),
    "--torch-model", TORCH, "--max-steps", "6", "--jsonl-answers", "--only-executed",
]
if SUB:
    cmd += ["--torch-subfolder", SUB]
print(" ".join(f'"{c}"' if " " in c else c for c in cmd))
# subprocess.run(cmd, cwd=REPO, check=True)
print("# Uncomment run_inference when ready (loads full model on each process start).")


## Baseline vs finetuned (shell)

From the repo root, for an apples-to-apples table:

```bash
# Quick: 5 episodes
LIMIT=5 bash compare_qwen17_eval.sh

# Use Hub for weights (no 7GB under repo): MODE=hub
MODE=hub LIMIT=30 bash compare_qwen17_eval.sh
```

Outputs: `outputs/eval_zeroshot_qwen1.7b.json` and `outputs/eval_finetuned_rl.json` — compare `safe_ship`, `repaired_actions`, `false_blocks`, and `total_budget_spent`.
